In [23]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader,ConcatDataset
from LSTM_databuilder import AssistSequenceDataset
from LSTM_model_train import AssistLSTM
import os

### video completion status


'''
Frame-level data
    ↓
Sliding window dataset
    ↓
LSTM(seq2one)
    ↓
urgency_pred + type_pred
    ↓
threshold
    ↓
assist trigger
'''

In [24]:
#device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### hyper param

In [31]:
epochs = 60
batch_size = 32
lr = 3e-4
window_size = 160
stride = 30
num_workers = 4

### load data

In [32]:
npz_dir = r"C:\Users\loy49\Desktop\REPO\LSTM_HRC\data\built_dataset"
npz_paths = [os.path.join(npz_dir, f) for f in os.listdir(npz_dir) if f.endswith(".npz")]
datasets = []

In [33]:
for npz_path in npz_paths:
    dataset = AssistSequenceDataset(npz_path=npz_path,
                                    window_size=120,
                                    stride=30,
                                    predict_offset=0,
                                    mode="seq2one",
                                    use_type=True)
    datasets.append(dataset)

loader = DataLoader(ConcatDataset(datasets), batch_size=32, shuffle=True, num_workers=4, pin_memory=(device.type == "cuda"), drop_last=False)
for x, y in loader:
    print("Batch X shape:", x.shape)  # should be [batch_size, window_size, feature_dim]
    print("Batch Y shape:", y.shape)  # should be [batch_size]
    break

Batch X shape: torch.Size([32, 120, 35])
Batch Y shape: torch.Size([32, 4])


### model

In [35]:
# get one batch to infer feature_dim and num_types
x_sample, y_step = next(iter(loader))
input_dim = x_sample.shape[2]

num_types = y_step.shape[1] 
print(f"Input dim: {input_dim}, Num types: {num_types}")

Input dim: 35, Num types: 4


In [36]:
model = AssistLSTM(
    input_dim=input_dim,
    hidden_dim=128,
    num_types=num_types
).to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)

ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

lambda_type = 1.0
lambda_urgency = 1.0

model.train()

AssistLSTM(
  (lstm): LSTM(35, 128, batch_first=True)
  (type_head): Linear(in_features=128, out_features=4, bias=True)
)

In [37]:
import torch.nn.functional as F

### train

In [39]:
for epoch in range(epochs):

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for x, y_step in loader:

        x = x.to(device)
        y_step = y_step.to(device)

        optimizer.zero_grad()

        step_pred = model(x)  # [B, num_steps]

        # ===== soft cross entropy =====？？？
        log_probs = F.log_softmax(step_pred, dim=1)
        loss = -(y_step * log_probs).sum(dim=1).mean()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ===== accuracy (hard version) =====
        preds = torch.argmax(step_pred, dim=1)
        y_hard = torch.argmax(y_step, dim=1)

        total_correct += (preds == y_hard).sum().item()
        total_samples += y_hard.size(0)

    acc = total_correct / total_samples

    print(f"Epoch {epoch+1}/{epochs} "
          f"| Loss: {total_loss/len(loader):.4f} "
          f"| Acc: {acc:.4f}")
    
print("Training complete.")

#save model
torch.save(model.state_dict(), "lstm_hrc.pth")
print("Model saved.")

Epoch 1/60 | Loss: 1.3209 | Acc: 0.3723
Epoch 2/60 | Loss: 1.2115 | Acc: 0.4454
Epoch 3/60 | Loss: 1.1806 | Acc: 0.4591
Epoch 4/60 | Loss: 1.1615 | Acc: 0.4704
Epoch 5/60 | Loss: 1.1420 | Acc: 0.4806
Epoch 6/60 | Loss: 1.1392 | Acc: 0.4857
Epoch 7/60 | Loss: 1.1286 | Acc: 0.4934
Epoch 8/60 | Loss: 1.1427 | Acc: 0.4663
Epoch 9/60 | Loss: 1.1664 | Acc: 0.4591
Epoch 10/60 | Loss: 1.1403 | Acc: 0.5020
Epoch 11/60 | Loss: 1.1130 | Acc: 0.5209
Epoch 12/60 | Loss: 1.1147 | Acc: 0.5281
Epoch 13/60 | Loss: 1.1135 | Acc: 0.5087
Epoch 14/60 | Loss: 1.0806 | Acc: 0.5133
Epoch 15/60 | Loss: 1.0703 | Acc: 0.5235
Epoch 16/60 | Loss: 1.0949 | Acc: 0.5092
Epoch 17/60 | Loss: 1.0633 | Acc: 0.5240
Epoch 18/60 | Loss: 1.0293 | Acc: 0.5373
Epoch 19/60 | Loss: 1.0334 | Acc: 0.5414
Epoch 20/60 | Loss: 1.0166 | Acc: 0.5363
Epoch 21/60 | Loss: 0.9972 | Acc: 0.5557
Epoch 22/60 | Loss: 0.9856 | Acc: 0.5751
Epoch 23/60 | Loss: 1.0236 | Acc: 0.5434
Epoch 24/60 | Loss: 1.0016 | Acc: 0.5490
Epoch 25/60 | Loss: 1.037